# ConRFT in simulation (Colab)
Runs ConRFT (consistency-policy fine-tuning of the Octo-small VLA) on the same simulated Franka cube-lift task, scripted expert, interventions and evaluation cubes as the HIL-SERL notebook, so both methods end up in one results table.

Stages (each is one queue entry, resumable, evaluated on 100 fixed episodes when it finishes):
- **SFT**: offline, demos only, behaviour cloning (Q weight 0, BC weight 1).
- **Cal-ConRFT**: offline, demos only, Cal-QL + BC (Q 0.1, BC 1).
- **HIL-ConRFT**: online with the scripted expert's interventions, starting from a pretrained checkpoint (Q 1, BC 0.1), 30k env steps at 10 Hz like HIL-SERL.
- Optional baselines: HG-DAgger (online BC on demos + interventions only), no interventions, fewer demos, BC weight, wide cube range.

**Runtime:** *Runtime → Change runtime type*: **A100 GPU** (recommended: online training keeps two copies of Octo, learner and actor), or L4 if the real-Octo test (cell 4) shows enough headroom. Turn on **High RAM**. Do not pick G4 (Blackwell): this setup uses jax 0.4.20 with CUDA 11 / cuDNN 8.

**Disconnects:** everything is kept in `MyDrive/hilserl-runs/` (the same folder as the HIL-SERL runs). Run cells 1-3 and the queue cell again; finished runs are skipped and an interrupted one resumes from its last checkpoint.

In [ ]:
import subprocess
name, mem_mib, cc = [x.strip() for x in subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"], text=True
).splitlines()[0].split(",")]
GPU_MEM_GB = int(mem_mib) / 1024
ram_gb = int(open("/proc/meminfo").read().split()[1]) / 1024**2
print(f"GPU: {name}, {GPU_MEM_GB:.0f} GB, compute capability {cc}; RAM: {ram_gb:.0f} GB")
if float(cc) >= 10:
    raise SystemExit(f"{name} is a Blackwell GPU (sm_{cc.replace('.', '')}). This setup pins jax 0.4.20 with CUDA 11 / cuDNN 8, "
                     "which cannot run on it. Switch the runtime to A100 or L4.")

## 1. Settings and Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/hilserl-runs"   # shared with the HIL-SERL runs
CONRFT_WORKDIR = "/content/conrft-sim-work"
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.environ.update(
    DRIVE_ROOT=DRIVE_ROOT, LOCAL_ROOT="/content/runs",
    WORKDIR=CONRFT_WORKDIR, CONRFT_WORKDIR=CONRFT_WORKDIR,
    # GPU memory caps (fractions of the card) for the learner and the actor; both hold Octo
    LEARNER_FRAC=".5" if GPU_MEM_GB >= 39 else ".55",
    ACTOR_FRAC=".3" if GPU_MEM_GB >= 39 else ".35",
    # replay buffer capacity: ~0.25 MB per transition (256 px + 128 px images, 2 frames)
    CONRFT_SIM_BUFFER="20000" if ram_gb >= 40 else "10000",
    CONRFT_SIM_CKPT_PERIOD="1000",
    MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
)
print("learner/actor GPU fractions:", os.environ["LEARNER_FRAC"], os.environ["ACTOR_FRAC"], "| replay buffer:", os.environ["CONRFT_SIM_BUFFER"])
print("HIL-SERL demos on Drive:", sorted(os.listdir(f"{DRIVE_ROOT}/demos")) if os.path.isdir(f"{DRIVE_ROOT}/demos") else "none (run the HIL-SERL notebook's demo cell first)")

## 2. Install (about 10 minutes)
Clones this repository, then `conrft-sim/setup.sh` clones ConRFT, its Octo fork and SERL at pinned commits, applies the fixes in `apply_patches.py`, adds the shared `sim_pick_cube` task and installs a Python 3.10 environment (jax 0.4.20 with CUDA 11, cuDNN 8.9, TensorFlow 2.15, Octo).

In [ ]:
%%bash
set -e
apt-get -qq install -y libegl1 libgl1 rsync > /dev/null
# Let EGL find the NVIDIA driver (as in MuJoCo's Colab tutorial); without this, MuJoCo renders on the CPU.
ICD=/usr/share/glvnd/egl_vendor.d/10_nvidia.json
if [ ! -f "$ICD" ]; then
  mkdir -p "$(dirname "$ICD")"
  printf '{\n  "file_format_version" : "1.0.0",\n  "ICD" : { "library_path" : "libEGL_nvidia.so.0" }\n}\n' > "$ICD"
fi
[ -d /content/seminar ] || git clone -q https://github.com/yigitbatki/seminar.git /content/seminar
git -C /content/seminar pull -q   # pick up fixes when the cell is re-run
bash /content/seminar/conrft-sim/setup.sh gpu "$CONRFT_WORKDIR" 2>&1 | tail -n 5

## 3. Check the GPU, the simulator and the renderer

In [ ]:
%%bash
source "$CONRFT_WORKDIR/.venv/bin/activate"
cd "$CONRFT_WORKDIR/conrft/examples"
python -c "import jax; print('jax devices:', jax.devices())"
python test_sim_env.py --episodes 3 2>&1 | grep -E "observation space|episode|success"

## 4. Real-Octo test (about 15-25 minutes; run once)
Downloads Octo-small, records 2 demos, pretrains 1,000 steps, then runs 300 online env steps with learner and actor. Prints the **peak GPU memory**, pretraining speed and actor speed. If it runs out of memory, lower `LEARNER_FRAC`/`ACTOR_FRAC` in cell 1 only if the peak is far below the card, otherwise switch to an A100. Use the speeds to estimate the queue's duration: a 20k-step pretraining takes 20,000 / (steps/s) seconds.

In [ ]:
!WORKDIR=$CONRFT_WORKDIR bash /content/seminar/conrft-sim/octo_test.sh

## 5. Experiment queue
Each entry changes only what it lists; everything else is the HIL-SERL run 2 setup (20 demos, 20% interventions, default cube range, 30k env steps at 10 Hz, 100 evaluation episodes on the same fixed cubes). ConRFT's demos are recorded once at run 2's 20 cube positions (about 15 minutes) and kept in `MyDrive/hilserl-runs/demos/conrft/`.

The default list is the core comparison: SFT and Cal-ConRFT (offline, 20k steps) for seeds 42-44, then HIL-ConRFT from each Cal-ConRFT checkpoint. Uncomment the optional entries for the baselines and ablations. Pretraining entries evaluate the offline policy; online entries the final online policy.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/seminar/hilserl")
import experiment_queue as eq
importlib.reload(eq)

C = dict(METHOD="conrft")
QUEUE = [
    {"RUN_NAME": "conrft_sft_s42", **C, "STAGE": "pretrain", "Q_WEIGHT": 0.0, "BC_WEIGHT": 1.0, "SEED": 42},
    {"RUN_NAME": "conrft_cal_s42", **C, "STAGE": "pretrain", "SEED": 42},
    {"RUN_NAME": "conrft_sft_s43", **C, "STAGE": "pretrain", "Q_WEIGHT": 0.0, "BC_WEIGHT": 1.0, "SEED": 43},
    {"RUN_NAME": "conrft_cal_s43", **C, "STAGE": "pretrain", "SEED": 43},
    {"RUN_NAME": "conrft_sft_s44", **C, "STAGE": "pretrain", "Q_WEIGHT": 0.0, "BC_WEIGHT": 1.0, "SEED": 44},
    {"RUN_NAME": "conrft_cal_s44", **C, "STAGE": "pretrain", "SEED": 44},
    {"RUN_NAME": "conrft_hil_s42", **C, "INIT_FROM": "conrft_cal_s42", "SEED": 42},
    {"RUN_NAME": "conrft_hil_s43", **C, "INIT_FROM": "conrft_cal_s43", "SEED": 43},
    {"RUN_NAME": "conrft_hil_s44", **C, "INIT_FROM": "conrft_cal_s44", "SEED": 44},
    # Optional baselines and ablations (seed 42):
    # {"RUN_NAME": "conrft_hgdagger_s42", **C, "INIT_FROM": "conrft_sft_s42", "Q_WEIGHT": 0.0, "BC_WEIGHT": 1.0, "BC_DEMO_ONLY": True},
    # {"RUN_NAME": "conrft_hil_from_sft_s42", **C, "INIT_FROM": "conrft_sft_s42"},
    # {"RUN_NAME": "conrft_hil_itv0_s42", **C, "INIT_FROM": "conrft_cal_s42", "INTERVENE_PROB": 0.0},
    # {"RUN_NAME": "conrft_cal_demos5_s42", **C, "STAGE": "pretrain", "N_DEMOS": 5},
    # {"RUN_NAME": "conrft_hil_demos5_s42", **C, "INIT_FROM": "conrft_cal_demos5_s42", "N_DEMOS": 5},
    # {"RUN_NAME": "conrft_hil_bc05_s42", **C, "INIT_FROM": "conrft_cal_s42", "BC_WEIGHT": 0.5},
    # {"RUN_NAME": "conrft_cal_wide_s42", **C, "STAGE": "pretrain", "CUBE_RANGE": "wide"},
    # {"RUN_NAME": "conrft_hil_wide_s42", **C, "INIT_FROM": "conrft_cal_wide_s42", "CUBE_RANGE": "wide"},
]
eq.run_queue(QUEUE)

## 6. Results of all runs (HIL-SERL and ConRFT)
One table for every finished run in `MyDrive/hilserl-runs/`: method and stage, success rate over the evaluation episodes with 95% Wilson confidence intervals (ConRFT's policy always samples, so it has only the sampled-action column), mean steps of the successful episodes, and for online runs the fraction of training episodes with a takeover; plus the policy-alone learning curves. Saved as `summary.csv` / `summary.png`.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/seminar/hilserl")
import experiment_queue as eq
importlib.reload(eq)
import pandas as pd
pd.set_option("display.max_colwidth", 70)
table, fig = eq.summarize()
display(table)